<a href="https://colab.research.google.com/github/lawcarp5-bit/steam-game-value-finder/blob/main/https_github_com_lawcarp5_bit_steam_game_value_finder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Steam Game Value Finder
My app helps gamers decide which games to buy by comparing different prices, the gameplay, and ratings.

In [1]:
import pandas as pd

# Load Steam games from GitHub so anyone can run the notebook.
url = "https://raw.githubusercontent.com/lawcarp5-bit/steam-game-value-finder/main/steam_games.csv"

games = pd.read_csv(url)

print("Loaded", len(games), "games")
print(games.head())

Loaded 1000 games
    appid                             name release_date  release_year  price  \
0     730                 Counter-Strike 2   2012-08-21          2012   0.00   
1  578080              PUBG: BATTLEGROUNDS   2017-12-21          2017   0.00   
2     570                           Dota 2   2013-07-09          2013   0.00   
3  271590        Grand Theft Auto V Legacy   2015-04-13          2015   0.00   
4  359550  Tom Clancy's Rainbow Six® Siege   2015-12-01          2015   3.99   

   discount  required_age  windows    mac  linux  ... positive negative  \
0         0             0     True  False   True  ...  7480813  1135108   
1         0             0     True  False  False  ...  1487960  1024436   
2         0             0     True   True   True  ...  1998462   451338   
3         0            17     True  False  False  ...  1719950   250012   
4        80            17     True  False  False  ...  1152763   218446   

  pct_pos_total  num_reviews_total       estimated

In [2]:
def best_value(games):
    best_game = ""
    best_score = 0

    for i in range(len(games)):
        price = games["price"][i]
        hours = games["avg_playtime_hours"][i]

        if price > 0:
            score = hours / price

            if score > best_score:
                best_score = score
                best_game = games["name"][i]

    return best_game

print(best_value(games))

Killing Floor 2


In [3]:
#Which action games have the best ratings regardless of price?
def best_action_games():
    action = games[games["genres"].str.contains("Action", na=False)]
    action = action.sort_values("pct_pos_total", ascending=False)
    return action

print(best_action_games()[["name", "pct_pos_total"]].head(10))

                          name  pct_pos_total
473  HoloCure - Save the Fans!             99
311                Katana ZERO             98
911              Split Fiction             98
63           Vampire Survivors             98
216            Half-Life: Alyx             98
54           People Playground             98
920                    VTOL VR             98
320                Pizza Tower             98
57                       Hades             98
352       BattleBlock Theater®             97


In [4]:
#Which games offer the most gameplay per dollar?
def best_value_games():
    paid = games[games["price"] > 0].copy()
    paid["value"] = paid["avg_playtime_hours"] / paid["price"]
    paid = paid.sort_values("value", ascending=False)
    return paid

print(best_value_games()[["name", "price", "avg_playtime_hours", "value"]].head(10))

                                name  price  avg_playtime_hours       value
220                  Killing Floor 2   2.99               546.2  182.675585
418                Age of History II   4.99               349.9   70.120240
255                   Cookie Clicker   4.99               308.7   61.863727
4    Tom Clancy's Rainbow Six® Siege   3.99               236.7   59.323308
579                    Bright Memory   7.99               346.7   43.391740
280              Company of Heroes 2   4.99               215.8   43.246493
254         FINAL FANTASY XIV Online  19.99               837.6   41.900950
8                        Garry's Mod   5.99               183.1   30.567613
26             ARK: Survival Evolved  14.99               217.8   14.529686
199                          Brotato   2.99                39.8   13.311037


In [5]:
#Which Strategy games have the highest player ratings?
def best_strategy_games():
    strategy = games[games["genres"].str.contains("Strategy", na=False)]
    strategy = strategy.sort_values("pct_pos_total", ascending=False)
    return strategy

print(best_strategy_games()[["name", "pct_pos_total"]].head(10))


                                  name  pct_pos_total
154                            Balatro             98
94                            RimWorld             97
87                        Satisfactory             97
98                            Factorio             97
43                         Bloons TD 6             97
107                     Slay the Spire             97
130  Totally Accurate Battle Simulator             97
241               Dyson Sphere Program             97
967      Stronghold Crusader HD (2012)             97
16                     Baldur's Gate 3             96


Outside Tool: CheapShark API

Question: What prices does CheapShark list for a game in my Steam dataset?

Purpose: I used CheapShark to look up retailer prices as an additional comparison to the downloaded file Steam-Games.


In [6]:
import requests

# Compares five Steam games using CheapShark prices to calculate gameplay hours per dollar.

names = ["Red Dead Online", "Grand Theft Auto", "Cyberpunk", "Elden Ring", "Hogwarts"]

for name in names:
    url = "https://www.cheapshark.com/api/1.0/games?title=" + name.replace(" ", "%20")
    response = requests.get(url, headers={"User-Agent": "SteamGameValueFinder/1.0"})

    print("\n", name)

    if response.status_code == 200:
        deals = response.json()
        match = games[games["name"].str.contains(name, case=False, na=False)]

        if len(match) > 0:
            for deal in deals:
                if deal["external"].lower() == match["name"].values[0].lower():
                    price = float(deal["cheapest"])
                    hours = match["avg_playtime_hours"].values[0]

                    if price > 0:
                        print("Price: $", price)
                        print("Hours per dollar:", round(hours / price, 2))
                    break
            else:
                print("No matching CheapShark game found")
        else:
            print("Not found in Steam dataset")
    else:
        print("CheapShark error:", response.status_code)


 Red Dead Online
Price: $ 19.99
Hours per dollar: 3.99

 Grand Theft Auto
No matching CheapShark game found

 Cyberpunk
Price: $ 59.99
Hours per dollar: 1.39

 Elden Ring
Price: $ 52.69
Hours per dollar: 3.72

 Hogwarts
Price: $ 5.99
Hours per dollar: 6.33
